# Prepare top-quark data for MATLAB

This notebook downloads the official **training** file, verifies its published checksum, and converts a subset for the MATLAB CNN / GraphSAGE / winner-inspired reference comparison.

It prepares data in Colab; MATLAB training runs in MATLAB R2024a+ with Deep Learning Toolbox or the repository's GitHub Actions runner. No GPU is needed for conversion. The source download is approximately 1.04 GB even for a small subset.

References: [dataset](https://doi.org/10.5281/zenodo.2603256), [2025 winner comparison](https://github.com/AstroAli5/top-quark-tagging-ai-challenge/blob/main/docs/WINNER_COMPARISON.md), [execution options](https://github.com/AstroAli5/top-quark-tagging-ai-challenge/blob/main/docs/PLATFORMS.md).


In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path('/content') if Path('/content').is_dir() else Path.cwd()
repo = root / 'top-quark-tagging-ai-challenge'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/AstroAli5/top-quark-tagging-ai-challenge.git',
                    str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(repo / 'requirements.txt')], check=True)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


## Download the verified source and choose a subset

Start with 2,000 jets. Raising this number increases MATLAB memory use and training time. The pipeline splits this subset 70%/15%/15%; the test portion is an internal holdout, not the official test partition.


In [ ]:
MAX_JETS = 2000
assert isinstance(MAX_JETS, int) and MAX_JETS > 0
source = repo / 'data' / 'train.h5'
output = repo / 'data' / 'jets_colab.mat'
subprocess.run([sys.executable, str(repo / 'scripts' / 'download_dataset.py'),
                '--output', str(source)], check=True)
subprocess.run([sys.executable, str(repo / 'scripts' / 'convert_dataset.py'),
                '--input', str(source), '--output', str(output),
                '--max-jets', str(MAX_JETS)], check=True)


The converter protects existing output files. To deliberately replace a prior conversion, add `--force` to the converter argument list and rerun that cell.


In [ ]:
import json
from scipy.io import loadmat

converted = loadmat(output)
provenance = json.loads(str(converted['provenance_json'].item()))
print(json.dumps(provenance, indent=2))
assert converted['particleData'].shape[0] == converted['labels'].shape[0]
print('Ready:', output.name, converted['particleData'].shape)


## Download and run in MATLAB

The next cell opens a browser download in Colab. Put `jets_colab.mat` in the MATLAB repository's `data/` folder. Then, from the repository folder in MATLAB:

```matlab
cfg = projectConfig;
cfg.inputFile = fullfile(cfg.dataDir,'jets_colab.mat');
cfg.cnnEpochs = 3;
cfg.graphEpochs = 3;
cfg.winnerEpochs = 3;
run_all(cfg)
run_winner_comparison(cfg)
```

Three epochs are a feasibility check. Report only your measured results, and preserve the configurations and source metadata with them. Use a larger training budget and several seeds for research conclusions.


In [ ]:
from google.colab import files
files.download(str(output))
